# Descriptive-Title
Felix Zaussinger | XX.YY.ZZZZ

Core Analysis Goal(s)
1.
2.
3.

Key Insight(s)
1.
2.
3.

In [142]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


#### Prepare unweighted shares at ISCO-08 3-digit level

Read final GBN classifications (ESCO-level)

In [143]:
df_sl = pd.read_csv(
    os.path.join(useful_paths.data_processed, "esco", "esco_level_gbn_classification_short_lists.csv"),
    index_col=0
)

df_sl_tobi = pd.read_csv(
    os.path.join(useful_paths.data_processed, "esco", "esco_level_gbn_classification_short_lists_tobi.csv"),
    index_col=0
)

Create ESCO-ISCO correspondance file

In [144]:
from src.data.framework import Esco
esco = Esco()
occ = esco.occupations
isco_col = "isco_code"
data_container = []

In [145]:
for n_digits_isco in [1, 2, 3, 4]:
    # Create ESCO-ISCO correspondance file
    occ[isco_col] = occ.iscoGroup.str[:n_digits_isco]
    esco_to_isco = pd.merge(occ, esco.isco_groups[["code", "preferredLabel"]], left_on=isco_col, right_on="code", how="left", suffixes=["_esco", "_isco"])
    esco_to_isco = esco_to_isco[["conceptUri", isco_col, "preferredLabel_isco"]]

    # Merge ISCO codes, calculate per-category count of ESCO occupations within ISCO groups
    df_sl_merged = df_sl.merge(esco_to_isco, on="conceptUri", how="left")
    df_sl_tobi_merged = df_sl_tobi.merge(esco_to_isco, on="conceptUri", how="left")

    df_sl_merged["n_esco"] = 1
    df_sl_merged_by_isco = df_sl_merged.groupby([isco_col, "preferredLabel_isco", "gbn_classification_short_list"])["n_esco"].count()
    df_sl_merged_by_isco = df_sl_merged_by_isco.reset_index()
    df_sl_merged_by_isco["isco_level"] = n_digits_isco

    df_sl_tobi_merged["n_esco"] = 1
    df_sl_tobi_merged_by_isco = df_sl_tobi_merged.groupby([isco_col, "preferredLabel_isco", "gbn_classification_short_list"])["n_esco"].count()
    df_sl_tobi_merged_by_isco = df_sl_tobi_merged_by_isco.reset_index()
    df_sl_tobi_merged_by_isco["isco_level"] = n_digits_isco

    # Calc unweighted shares at ISCO-08 3-digit level: df_sl
    cnt = 0
    data_store = []
    cols_all = {"green", "brown", "neutral"}

    for grp, grp_df in df_sl_merged_by_isco.groupby(isco_col):

        # pivot long to wide, simplify multi-indices
        grp_df_piv = grp_df.pivot(index=["isco_level", isco_col, "preferredLabel_isco"], columns=["gbn_classification_short_list"], values=["n_esco"])
        grp_df_piv.columns = grp_df_piv.columns.get_level_values(1)
        grp_df_piv = grp_df_piv.reset_index()
        #grp_df_piv.index = grp_df_piv.index.get_level_values(0)

        # create missing column
        cols_exist = set(grp_df_piv.columns.tolist())
        col_missing = list(cols_all - cols_exist)
        grp_df_piv[col_missing] = 0

        # calc shares
        grp_df_piv["N"] = grp_df_piv[list(cols_all)].sum(axis=1)

        for col in list(cols_all):
            grp_df_piv["share_{}".format(col)] = grp_df_piv[col] / grp_df_piv["N"]

        # append
        data_store.append(grp_df_piv)

        # concat to new df
        df_out = pd.concat(data_store)

        # save individual file
        df_out.to_csv(
            os.path.join(useful_paths.data_processed, "esco", "short_list_gbn_shares_by_isco{}d_unweighted.csv".format(n_digits_isco))
        )

        # flag list version
        df_out["list_version"] = "short_list"

        # global append
        data_container.append(df_out)

    # Calc unweighted shares at ISCO-08 3-digit level: df_sl_tobi
    cnt = 0
    data_store = []
    cols_all = {"green", "brown", "neutral"}

    for grp, grp_df in df_sl_tobi_merged_by_isco.groupby(isco_col):

        # pivot long to wide, simplify multi-indices
        grp_df_piv = grp_df.pivot(index=["isco_level", isco_col, "preferredLabel_isco"], columns=["gbn_classification_short_list"], values=["n_esco"])
        grp_df_piv.columns = grp_df_piv.columns.get_level_values(1)
        grp_df_piv = grp_df_piv.reset_index()
        #grp_df_piv.index = grp_df_piv.index.get_level_values(0)

        # create missing column
        cols_exist = set(grp_df_piv.columns.tolist())
        col_missing = list(cols_all - cols_exist)
        grp_df_piv[col_missing] = 0

        # calc shares
        grp_df_piv["N"] = grp_df_piv[list(cols_all)].sum(axis=1)

        for col in list(cols_all):
            grp_df_piv["share_{}".format(col)] = grp_df_piv[col] / grp_df_piv["N"]

        # append
        data_store.append(grp_df_piv)

    # concat to new df
    df_out = pd.concat(data_store)

    # save individual file
    df_out.to_csv(
        os.path.join(useful_paths.data_processed, "esco", "short_list_tobi_gbn_shares_by_isco{}d_unweighted.csv".format(n_digits_isco))
    )

    # flag list version
    df_out["list_version"] = "short_list_tobi"

    # global append
    data_container.append(df_out)

Combine to single long DF and save (more compact than single csvs)

In [151]:
gbn_shares_final = pd.concat(data_container).drop_duplicates()
gbn_shares_final = gbn_shares_final.sort_values(["list_version", "isco_level"], ascending=True)
gbn_shares_final = gbn_shares_final.reset_index(drop=True)

utils.save_df_to_files(
    df=gbn_shares_final,
    output_dir=os.path.join(useful_paths.data_processed, "esco"),
    fname_no_ext="final_gbn_shares_by_isco_unweighted"
)

In [150]:
gbn_shares_final

gbn_classification_short_list,isco_level,isco_code,preferredLabel_isco,neutral,brown,green,N,share_neutral,share_brown,share_green,list_version
0,1,0,Armed forces occupations,21,0,0,21,1.000000,0.000000,0.000000,short_list
1,1,1,Managers,331,4,3,338,0.979290,0.011834,0.008876,short_list
2,1,2,Professionals,782,25,39,846,0.924350,0.029551,0.046099,short_list
3,1,3,Technicians and associate professionals,605,12,36,653,0.926493,0.018377,0.055130,short_list
4,1,4,Clerical support workers,87,2,0,89,0.977528,0.022472,0.000000,short_list
...,...,...,...,...,...,...,...,...,...,...,...
1201,4,9613,Sweepers and related labourers,1,0,0,1,1.000000,0.000000,0.000000,short_list_tobi
1202,4,9621,"Messengers, package deliverers and luggage por...",2,0,0,2,1.000000,0.000000,0.000000,short_list_tobi
1203,4,9622,Odd job persons,1,0,0,1,1.000000,0.000000,0.000000,short_list_tobi
1204,4,9623,Meter readers and vending-machine collectors,2,0,0,2,1.000000,0.000000,0.000000,short_list_tobi
